In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

BASE_PATH = "dbfs:/Volumes/workspace/landing/inputs"

print("Arquivos disponíveis na Landing:")
for arquivo in dbutils.fs.ls(BASE_PATH):
    print("-", arquivo.name)


Arquivos disponíveis na Landing:
- cotacao_dolar.json
- cotacao_dolar_11.json
- credits_and_tags_IMDB_TMDB.csv
- movies_financials_IMDB_TMDB.csv
- movies_info_TMDB_IMDB.csv
- movies_metrics_IMDB_TMDB.csv
- movies_reviews.csv


## Proteção para reexecução

O requisito da Bronze é utilizar `append`. Como os arquivos de origem deste projeto são estáticos, a função abaixo evita inserir novamente o mesmo lote caso o notebook seja reexecutado durante testes ou pelo Workflow. Em uma primeira execução, a escrita é feita normalmente em modo `append`.


In [0]:
def gravar_append_sem_duplicar(df, tabela):
    qtd_origem = df.count()

    if spark.catalog.tableExists(tabela):
        qtd_destino = spark.table(tabela).count()

        if qtd_destino == qtd_origem:
            print(f"{tabela}: lote já carregado ({qtd_destino} linhas). Escrita ignorada para evitar duplicação.")
            return

        raise Exception(
            f"{tabela}: quantidade existente ({qtd_destino}) difere da origem atual ({qtd_origem}). "
            "Interrompendo para evitar append duplicado ou inconsistente."
        )

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(tabela)
    )

    print(f"{tabela}: {qtd_origem} linhas gravadas em Delta com append.")


## 1. movies_info_TMDB_IMDB.csv → bronze.tb_movies_info


In [0]:
caminho_info = f"{BASE_PATH}/movies_info_TMDB_IMDB.csv"

df_info_filmes = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .csv(caminho_info)
    .withColumn("ingestion_datetime", F.current_timestamp())
)

gravar_append_sem_duplicar(df_info_filmes, "bronze.tb_movies_info")
display(df_info_filmes.limit(10))


bronze.tb_movies_info: lote já carregado (106930 linhas). Escrita ignorada para evitar duplicação.


id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-09-21T19:25:00.488Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-09-21T19:25:00.488Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-09-21T19:25:00.488Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-09-21T19:25:00.488Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-09-21T19:25:00.488Z
284054,tt1825683,Black Panther,Black Panther,en,2018-02-13,135,Released,"King T'Challa returns home to the reclusive, technologically advanced African nation of Wakanda to serve as his country's new leader. However, T'Challa soon finds that he is challenged for the throne by factions within his own country as well as without. Using powers reserved to Wakandan kings, T'Challa assumes the Black Panther mantle to join with ex-girlfriend Nakia, the queen-mother, his princess-kid sister, members of the Dora Milaje (the Wakandan 'special forces') and an American secret agent, to prevent Wakanda from being dragged into a world war.",null,2026-09-21T19:25:00.488Z
284052,tt1211837,Doctor Strange,Doctor Strange,en,2016-10-25,115,Released,"After his career is destroyed, a brilliant but arrogant surgeon gets a new lease on life when a sorcerer takes him under her wing and trains him to defend the world against evil.",The impossibilities are endless.,2026-09-21T19:25:00.488Z
315635,tt2250912,Spider-Man: Homecoming,Spider-Man: Homecoming,en,2017-07-05,133,RELEASED,"Following the events of Captain America: Civil War, Peter Parker, with the help of his mentor Tony Stark, tries to balance his life as an ordinary high school student in Queens, New York City, with fighting crime as his superhero alter ego Spider-Man as a new threat, the Vulture, emerges.",Homework can wait. The city can't.,2026-09-21T19:25:00.488Z
283995,tt3896198,Guardians of the Galaxy Vol. 2,Guardians of the Galaxy Vol. 2,en,2017-04-19,137,Released,The Guardians must fight to keep their newfound family together as t

## 2. movies_financials_IMDB_TMDB.csv → bronze.tb_movies_financials


In [0]:
caminho_financials = f"{BASE_PATH}/movies_financials_IMDB_TMDB.csv"

df_financials = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .csv(caminho_financials)
    .withColumn("ingestion_datetime", F.current_timestamp())
)

gravar_append_sem_duplicar(df_financials, "bronze.tb_movies_financials")
display(df_financials.limit(10))


bronze.tb_movies_financials: lote já carregado (106165 linhas). Escrita ignorada para evitar duplicação.


id,budget,revenue,ingestion_datetime
293660,58000000,Unknown,2026-09-21T19:25:03.434Z
299536,300000000,2052415039,2026-09-21T19:25:03.434Z
299534,356000000,2800000000,2026-09-21T19:25:03.434Z
475557,55000000,1074458282,2026-09-21T19:25:03.434Z
271110,250000000,Não Informado,2026-09-21T19:25:03.434Z
284054,200000000,1349926083,2026-09-21T19:25:03.434Z
284052,180000000,676343174,2026-09-21T19:25:03.434Z
315635,175000000,880166924,2026-09-21T19:25:03.434Z
283995,200000000,863756051,2026-09-21T19:25:03.434Z
297761,175000000,746846894,2026-09-21T19:25:03.434Z


## 3. movies_metrics_IMDB_TMDB.csv → bronze.tb_movies_metrics


In [0]:
caminho_metrics = f"{BASE_PATH}/movies_metrics_IMDB_TMDB.csv"

df_metrics = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .csv(caminho_metrics)
    .withColumn("ingestion_datetime", F.current_timestamp())
)

gravar_append_sem_duplicar(df_metrics, "bronze.tb_movies_metrics")
display(df_metrics.limit(10))


bronze.tb_movies_metrics: lote já carregado (107364 linhas). Escrita ignorada para evitar duplicação.


id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-09-21T19:25:06.536Z
299536,"154,34",8.255,27713,8.4,1406782,2026-09-21T19:25:06.536Z
299534,91.756,8.263,23857,8.4,1484150,2026-09-21T19:25:06.536Z
475557,"54,522",8.168,23425,8.3,1723035,2026-09-21T19:25:06.536Z
271110,70.741,7.4,21541,7.8,947222,2026-09-21T19:25:06.536Z
284054,43.665,7.39,null,7.3,924922,2026-09-21T19:25:06.536Z
284052,70.535,7.427,20935,7.5,895880,2026-09-21T19:25:06.536Z
315635,65.88,7.345,20507,7.4,835116,2026-09-21T19:25:06.536Z
283995,67.553,7.624,20353,7.6,844767,2026-09-21T19:25:06.536Z
297761,35.356,5.909,20097,5.9,null,2026-09-21T19:25:06.536Z


## 4. credits_and_tags_IMDB_TMDB.csv → bronze.tb_credits_and_tags


In [0]:
caminho_credits = f"{BASE_PATH}/credits_and_tags_IMDB_TMDB.csv"

df_credits_tags = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .csv(caminho_credits)
    .withColumn("ingestion_datetime", F.current_timestamp())
)

gravar_append_sem_duplicar(df_credits_tags, "bronze.tb_credits_and_tags")
display(df_credits_tags.limit(10))


bronze.tb_credits_and_tags: lote já carregado (106320 linhas). Escrita ignorada para evitar duplicação.


id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_datetime
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-09-21T19:25:09.936Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-09-21T19:25:09.936Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-09-21T19:25:09.936Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-09-21T19:25:09.936Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-09-21T19:25:09.936Z
284054,"Action, Adventure, Science Fiction",Marvel Studios,United States of America,"English, Korean, Swahili, Xhosa","africa, superhero, based on comic, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Ryan Coogler,"Ryan Coogler, Joe Robert Cole, Stan Lee, Jack Kirby","Chadwick Boseman, Michael B. Jordan, Lupita Nyong'o, Danai Gurira, Martin Freeman, Daniel Kaluuya, Letitia Wright, Winston Duke, Sterling K. Brown, Angela Bassett",2026-09-21T19:25:09.936Z
284052,"Action, Adventure, Fantasy",Marvel Studios,UNITED STATES OF AMERICA,English,"magic, superhero, training, time, based on comic, sorcerer, doctor, neurosurgeon, wizard, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Scott Derrickson,N/A,"Benedict Cumberbatch, Chiwetel Ejiofor, Rachel McAdams, Benedict Wong, Mads Mikkelsen, Tilda Swinton, Michael Stuhlbarg, Benjamin Bratt, Scott Adkins, Zara Phythian",2026-09-21T19:25:09.936Z
315635,"Action, Adventure, Science Fiction, Drama","Ma

## 5. movies_reviews.csv → bronze.tb_movies_reviews


In [0]:
caminho_reviews = f"{BASE_PATH}/movies_reviews.csv"

df_reviews = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .csv(caminho_reviews)
    .withColumn("ingestion_datetime", F.current_timestamp())
)

gravar_append_sem_duplicar(df_reviews, "bronze.tb_movies_reviews")
display(df_reviews.limit(10))


bronze.tb_movies_reviews: lote já carregado (32412 linhas). Escrita ignorada para evitar duplicação.


id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-09-21T19:25:12.976Z
637007,Lucas Reis 602,3.9,null,2026-09-21T19:25:12.976Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-09-21T19:25:12.976Z
413036,Gabriela Monteiro 401,7.5,null,2026-09-21T19:25:12.976Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-09-21T19:25:12.976Z
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo.",2026-09-21T19:25:12.976Z
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo.",2026-09-21T19:25:12.976Z
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.,2026-09-21T19:25:12.976Z
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.,2026-09-21T19:25:12.976Z
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular.",2026-09-21T19:25:12.976Z


## 6. Cotação do dólar — API PTAX/Banco Central

Os widgets mantêm as datas parametrizadas no formato `MM-DD-AAAA`. A URL abaixo é a URL utilizada para obter o JSON bruto externamente.


In [0]:
dbutils.widgets.text("data_inicio", "09-12-2026", "Data inicial (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", "09-18-2026", "Data final (MM-DD-AAAA)")

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

url_bcb = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/"
    "odata/CotacaoDolarPeriodo"
    "(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra"
    "&$format=json"
)

print("Data inicial:", data_inicio)
print("Data final:", data_fim)
print("URL PTAX:", url_bcb)


Data inicial: 09-12-2026
Data final: 09-18-2026
URL PTAX: https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='09-12-2026'&@dataFinalCotacao='09-18-2026'&$select=dataHoraCotacao,cotacaoCompra&$format=json


### Ingestão do retorno bruto da API

Foram mantidos na Landing os arquivos JSON obtidos da API. O padrão `cotacao_dolar*.json` também inclui a cotação do último dia útil anterior ao fim de semana, necessária posteriormente para demonstrar o Forward Fill na Silver.


In [0]:
df_json_cotacao = (
    spark.read
    .option("multiLine", "true")
    .json(f"{BASE_PATH}/cotacao_dolar*.json")
)

df_cotacao = (
    df_json_cotacao
    .select(F.explode(F.col("value")).alias("cotacao"))
    .select(
        F.col("cotacao.dataHoraCotacao").alias("dataHoraCotacao"),
        F.col("cotacao.cotacaoCompra").alias("cotacaoCompra")
    )
    .dropDuplicates(["dataHoraCotacao", "cotacaoCompra"])
    .withColumn("ingestion_datetime", F.current_timestamp())
)

gravar_append_sem_duplicar(df_cotacao, "bronze.tb_cotacao_dolar")
display(df_cotacao.orderBy("dataHoraCotacao"))


bronze.tb_cotacao_dolar: lote já carregado (6 linhas). Escrita ignorada para evitar duplicação.


dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-09-11 13:07:22.532196,5.0912,2026-09-21T19:25:15.806Z
2026-09-14 13:10:08.144425,5.169,2026-09-21T19:25:15.806Z
2026-09-15 13:09:19.199664,5.1484,2026-09-21T19:25:15.806Z
2026-09-16 13:05:30.35873,5.152,2026-09-21T19:25:15.806Z
2026-09-17 13:03:21.858212,5.1515,2026-09-21T19:25:15.806Z
2026-09-18 13:03:34.742036,5.1569,2026-09-21T19:25:15.806Z


## Validação final da camada Bronze


In [0]:
validacoes = [
    ("movies_info_TMDB_IMDB.csv", "bronze.tb_movies_info"),
    ("movies_financials_IMDB_TMDB.csv", "bronze.tb_movies_financials"),
    ("movies_metrics_IMDB_TMDB.csv", "bronze.tb_movies_metrics"),
    ("credits_and_tags_IMDB_TMDB.csv", "bronze.tb_credits_and_tags"),
    ("movies_reviews.csv", "bronze.tb_movies_reviews"),
]

for arquivo, tabela in validacoes:
    qtd_csv = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{BASE_PATH}/{arquivo}")
        .count()
    )
    qtd_bronze = spark.table(tabela).count()

    status = "OK" if qtd_csv == qtd_bronze else "DIFERENTE"
    print(f"{tabela}: CSV={qtd_csv} | Bronze={qtd_bronze} | {status}")

print("bronze.tb_cotacao_dolar:", spark.table("bronze.tb_cotacao_dolar").count(), "linhas")


bronze.tb_movies_info: CSV=106930 | Bronze=106930 | OK
bronze.tb_movies_financials: CSV=106165 | Bronze=106165 | OK
bronze.tb_movies_metrics: CSV=107364 | Bronze=107364 | OK
bronze.tb_credits_and_tags: CSV=106320 | Bronze=106320 | OK
bronze.tb_movies_reviews: CSV=32412 | Bronze=32412 | OK
bronze.tb_cotacao_dolar: 6 linhas


In [0]:
print(
    "Linhas na bronze.tb_cotacao_dolar:",
    spark.table("bronze.tb_cotacao_dolar").count()
)

Linhas na bronze.tb_cotacao_dolar: 6
